# Antoine Parameter Determination

The Antoine equation relates saturated vapour pressure to temperature:

$$\log_{10} P = A - \frac{B}{C + T}$$

These parameters are typically obtained from experimental vapour pressure data through regression. However, experimental data is not always available — especially for novel molecules generated by CAMD.

This tutorial demonstrates how CAMD-ISS determines Antoine parameters from **group-contribution estimated properties** using three different methods, each requiring different levels of input data.

## Setup

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
sys.path.append(os.path.dirname(os.getcwd()))

from src.pure import PropertyCalculator, AntoineCalculator

pure_calc = PropertyCalculator()
antoine_calc = AntoineCalculator()

## Theoretical Background

From group-contribution models, we can estimate several pure component properties that locate distinct points on the vapour pressure curve:

| Property | Symbol | Description |
|----------|--------|-------------|
| Boiling point | $T_b$ | $P_{sat}(T_b) = 1.01325$ bar |
| Critical temperature | $T_c$ | $P_{sat}(T_c) = P_c$ |
| Critical pressure | $P_c$ | Pressure at critical point |
| Acentric factor | $\omega$ | Defines $P_{sat}(0.7 T_c) = 10^{-(\omega+1)} P_c$ |
| Enthalpy of vaporization | $\Delta H_v$ | Enables Clausius-Clapeyron estimate at 298.15 K |

With three points on the $\log P$ vs $T$ curve, we can fit the three Antoine parameters.

The software uses the **selected points method** from [Thomson (1946)](https://doi.org/10.1021/cr60119a001) to provide initial estimates, then solves the system using Pyomo/IPOPT.

For three saturated vapour pressure points $(y_1, y_2, y_3)$ at the corresponding temperatures $(t_1, t_2, t_3)$, the initialization is:

\begin{align}
\dfrac{y_3-y_2}{y_2-y_1} \dfrac{t_2-t_1}{t_3-t_2} &= 1 - \frac{t_3-t_1}{t_3 + C} \\
B &= \frac{y_3-y_1}{t_3 - t_1} (t_1+C)(t_3+C) \\
A &= y_2 + \frac{B}{t_2 + C}
\end{align}

## The Three Methods

CAMD-ISS provides three methods for Antoine parameter determination. All share the first two points — $(T_b, P_0)$ and $(T_c, P_c)$ — and differ in how the third constraint is obtained:

| Method | Required Properties | Third Point | Approach |
|--------|-------------------|-------------|----------|
| `acentric_factor` | $T_c$, $P_c$, $T_b$, $\omega$ | $P_{sat}(0.7 T_c) = 10^{-(\omega+1)} P_c$ | Optimization (Pyomo/IPOPT) |
| `enthalpy_vap` | $T_c$, $P_c$, $T_b$, $\Delta H_v$ | $P_{298} = P_0 \exp\left(\frac{\Delta H_v}{R}\left(\frac{1}{T_b} - \frac{1}{298.15}\right)\right)$ | Optimization (Pyomo/IPOPT) |
| `minimal` | $T_c$, $P_c$, $T_b$ | None — sets $C = 0$ | Analytical (2-parameter) |

## Example: Water

First, compute pure properties using the Constantinou-Gani group contribution method.

In [ ]:
compound = {'H2O': 1}

# Calculate all properties needed by any method
props = pure_calc.calculate(compound, properties=['tc', 'pc', 'tb', 'ac', 'hv'])

print("Estimated pure component properties (water):")
print(f"  Tc = {props['tc']:.2f} K")
print(f"  Pc = {props['pc']:.2f} bar")
print(f"  Tb = {props['tb']:.2f} K")
print(f"  ac = {props['ac']:.4f}")
print(f"  Hv = {props['hv']:.2f} kJ/mol")

### Method 1: Acentric Factor

Uses the definition of the acentric factor to obtain the third point:
$$P_{sat}(0.7 T_c) = 10^{-(\omega + 1)} \cdot P_c$$

In [ ]:
res_ac = antoine_calc.calculate_antoine_parameters(compound, critical_properties=props, method='acentric_factor')

print(f"Method: acentric_factor")
print(f"  A = {res_ac['A']:.4f}")
print(f"  B = {res_ac['B']:.2f}")
print(f"  C = {res_ac['C']:.2f}")
print(f"  Status: {res_ac['termination_condition']}")

### Method 2: Enthalpy of Vaporization

Uses the Clausius-Clapeyron equation with $\Delta H_v$ to estimate $P_{sat}$ at 298.15 K:
$$P_{298} = P_0 \cdot \exp\left(\frac{\Delta H_v}{R} \left(\frac{1}{T_b} - \frac{1}{298.15}\right)\right)$$

In [ ]:
res_hv = antoine_calc.calculate_antoine_parameters(compound, critical_properties=props, method='enthalpy_vap')

print(f"Method: enthalpy_vap")
print(f"  A = {res_hv['A']:.4f}")
print(f"  B = {res_hv['B']:.2f}")
print(f"  C = {res_hv['C']:.2f}")
print(f"  Status: {res_hv['termination_condition']}")

### Method 3: Minimal Data

When only $T_c$, $P_c$, and $T_b$ are available, the system is underdetermined for three parameters. This method sets $C = 0$ and solves analytically:

$$B = \frac{(\log P_0 - \log P_c) \cdot T_b \cdot T_c}{T_b - T_c}, \qquad A = \log P_0 + \frac{B}{T_b}$$

In [ ]:
res_min = antoine_calc.calculate_antoine_parameters(compound, critical_properties=props, method='minimal')

print(f"Method: minimal")
print(f"  A = {res_min['A']:.4f}")
print(f"  B = {res_min['B']:.2f}")
print(f"  C = {res_min['C']:.2f}  (fixed)")
print(f"  Status: {res_min['solver_status']}")

## Comparison of Methods

### Parameter Summary

In [ ]:
methods = {
    'acentric_factor': res_ac,
    'enthalpy_vap': res_hv,
    'minimal': res_min
}

print(f"{'Method':<20} {'A':>10} {'B':>10} {'C':>10}")
print("-" * 52)
for name, res in methods.items():
    print(f"{name:<20} {res['A']:>10.4f} {res['B']:>10.2f} {res['C']:>10.2f}")

# NIST reference for water (Stull, 1947): valid 344-373 K
print(f"\n{'NIST (Stull 1947)':<20} {'5.0835':>10} {'1663.13':>10} {'-45.62':>10}")

### Validation at Key Points

Each method should reproduce the known vapour pressure at the boiling point ($P_{sat}(T_b) \approx 1.01325$ bar) and at the critical point ($P_{sat}(T_c) \approx P_c$).

In [ ]:
print(f"{'Method':<20} {'P(Tb) [bar]':>12} {'err_Tb [%]':>12} {'P(Tc) [bar]':>12} {'err_Tc [%]':>12}")
print("-" * 70)
for name, res in methods.items():
    val = antoine_calc.validate_antoine_parameters(
        A=res['A'], B=res['B'], C=res['C'],
        tb=props['tb'], tc=props['tc'], pc=props['pc']
    )
    err_tb = val['error_at_boiling_point'] / 1.01325 * 100
    err_tc = val['relative_error_critical']
    print(f"{name:<20} {val['pressure_at_boiling_point']:>12.5f} {err_tb:>11.3f}% {val['pressure_at_critical_temp']:>12.3f} {err_tc:>11.3f}%")

### Vapour Pressure Curves

Comparison of $P_{sat}(T)$ from each method over the full range $T_b$ to $T_c$.

In [ ]:
T_range = np.linspace(props['tb'], props['tc'], 500)

colors = {'acentric_factor': 'C0', 'enthalpy_vap': 'C1', 'minimal': 'C2'}
labels = {'acentric_factor': 'Acentric factor', 'enthalpy_vap': 'Enthalpy of vap.', 'minimal': 'Minimal (C=0)'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for name, res in methods.items():
    P_calc = antoine_calc.antoine_correlation(res['A'], res['B'], res['C'], T_range)
    ax1.plot(T_range, P_calc, color=colors[name], label=labels[name])
    ax2.semilogy(1/T_range, P_calc, color=colors[name], label=labels[name])

# Mark key points
ax1.axhline(y=1.01325, color='gray', linestyle='--', alpha=0.5, label='$P_0$ = 1.013 bar')
ax1.axhline(y=props['pc'], color='gray', linestyle=':', alpha=0.5, label=f'$P_c$ = {props["pc"]:.1f} bar')
ax1.scatter([props['tb']], [1.01325], color='k', zorder=5, s=40)
ax1.scatter([props['tc']], [props['pc']], color='k', zorder=5, s=40)

ax1.set_xlabel('Temperature [K]')
ax1.set_ylabel('$P_{sat}$ [bar]')
ax1.set_title('Vapour Pressure Curve')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

ax2.set_xlabel('1/T [1/K]')
ax2.set_ylabel('$P_{sat}$ [bar]')
ax2.set_title('Clausius-Clapeyron Plot (log scale)')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Monotonicity Check

A physically meaningful $P_{sat}(T)$ must be strictly increasing. Let's verify this for each method.

In [ ]:
print(f"{'Method':<20} {'Monotonic':>10} {'min(dP/dT)':>15}")
print("-" * 47)
for name, res in methods.items():
    P_calc = antoine_calc.antoine_correlation(res['A'], res['B'], res['C'], T_range)
    dP = np.diff(P_calc)
    is_monotonic = np.all(dP > 0)
    print(f"{name:<20} {'Yes' if is_monotonic else 'NO':>10} {np.min(dP):>15.6f}")

## Example: Ethylene Glycol

Repeating the comparison for a high-boiling compound to see how the methods differ.

In [ ]:
compound_eg = {'DOH': 1}
props_eg = pure_calc.calculate(compound_eg, properties=['tc', 'pc', 'tb', 'ac', 'hv'])

print("Estimated properties (ethylene glycol):")
for k, v in props_eg.items():
    print(f"  {k} = {v:.4f}")

results_eg = {}
for method in ['acentric_factor', 'enthalpy_vap', 'minimal']:
    res = antoine_calc.calculate_antoine_parameters(compound_eg, critical_properties=props_eg, method=method)
    if res:
        results_eg[method] = res

print(f"\n{'Method':<20} {'A':>10} {'B':>10} {'C':>10}")
print("-" * 52)
for name, res in results_eg.items():
    print(f"{name:<20} {res['A']:>10.4f} {res['B']:>10.2f} {res['C']:>10.2f}")

In [ ]:
T_range_eg = np.linspace(props_eg['tb'], props_eg['tc'], 500)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for name, res in results_eg.items():
    P_calc = antoine_calc.antoine_correlation(res['A'], res['B'], res['C'], T_range_eg)
    ax1.plot(T_range_eg, P_calc, color=colors[name], label=labels[name])
    ax2.semilogy(1/T_range_eg, P_calc, color=colors[name], label=labels[name])

ax1.axhline(y=1.01325, color='gray', linestyle='--', alpha=0.5)
ax1.axhline(y=props_eg['pc'], color='gray', linestyle=':', alpha=0.5)
ax1.scatter([props_eg['tb']], [1.01325], color='k', zorder=5, s=40)
ax1.scatter([props_eg['tc']], [props_eg['pc']], color='k', zorder=5, s=40)

ax1.set_xlabel('Temperature [K]')
ax1.set_ylabel('$P_{sat}$ [bar]')
ax1.set_title('Ethylene Glycol')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

ax2.set_xlabel('1/T [1/K]')
ax2.set_ylabel('$P_{sat}$ [bar]')
ax2.set_title('Ethylene Glycol (Clausius-Clapeyron)')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

| Method | Pros | Cons |
|--------|------|------|
| `acentric_factor` | Three well-defined points, good extrapolation | Requires acentric factor estimation |
| `enthalpy_vap` | Uses readily available $\Delta H_v$, good near $T_b$ | Clausius-Clapeyron approximation may be poor far from $T_b$ |
| `minimal` | Only needs $T_c$, $P_c$, $T_b$ — always available | $C=0$ reduces flexibility, poorest fit |

The `enthalpy_vap` method is used as the default in the CAMD-ISS pipeline, as $\Delta H_v$ is estimated alongside other properties by the Constantinou-Gani method.